# Step 01: Trainingsmoeglichkeiten

Dieses Notebook stellt nur vor, wie die unterschiedlichen Modellfamilien trainiert werden koennen. Es fuehrt keine Evaluation aus und startet standardmaessig kein Training. Die komplette Bewertung liegt in `step06_evaluation_lab.ipynb`.

## Rollen Der Schritte

| Schritt | Aufgabe |
|---|---|
| Step01 | Trainingsvarianten und Beispielaufrufe erklaeren |
| Step02 | YOLO/Ultralytics trainieren |
| Step03 | Torchvision-Detektoren trainieren: Faster R-CNN, RetinaNet, FCOS |
| Step06 | Evaluation als Skript erzeugen |
| step06_evaluation_lab | Evaluation interpretieren, Tabellen/Grafiken/Ressourcen darstellen |
| Step10 | Referenzindex ohne Dopplungen |

In [ ]:
from pathlib import Path
import os
import sys


def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    candidates = [current, *current.parents]

    for candidate in candidates:
        lab = candidate / "face_model_lab"
        if (lab / "step00_common.py").exists():
            return candidate

    for candidate in candidates:
        if (candidate / "step00_common.py").exists():
            return candidate.parent

    raise FileNotFoundError(
        "Projektroot nicht gefunden. Starte das Notebook im Projektordner "
        "oder setze FACE_LAB_ROOT."
    )


ROOT = Path(os.environ["FACE_LAB_ROOT"]).expanduser().resolve() if os.environ.get("FACE_LAB_ROOT") else find_project_root()
LAB = ROOT / "face_model_lab"
PYTHON = Path(os.environ.get("FACE_LAB_PYTHON", sys.executable)).expanduser().resolve()


def show_command(parts):
    print(" ".join(str(p) for p in parts))


print("Projektroot:", ROOT)
print("Face Model Lab:", LAB)
print("Python:", PYTHON)

## Gemeinsame Parameter

Die Projektlaeufe nutzten eine AMD Radeon PRO W7800 48GB ueber ROCm/CUDA-API. Batch Size 2 wurde gewaehlt, weil die Modelle stabil auf der GPU laufen und die Datenbasis vergleichbar bleibt.

In [ ]:
BATCH_SIZE = 2
IMGSZ = 768
WORKERS = 0

# Breiter Smoke-Vergleich: alle Modellfamilien kurz testen.
SMOKE_REDUCTION = 20
SMOKE_EPOCHS = 1

# Fairerer Kandidatenvergleich: die zwei Favoriten laenger trainieren.
DEEP_REDUCTION = 6
DEEP_EPOCHS = 10

print('Smoke:', SMOKE_REDUCTION, 'Reduction,', SMOKE_EPOCHS, 'Epoche')
print('Kandidatenlauf:', DEEP_REDUCTION, 'Reduction,', DEEP_EPOCHS, 'Epochen')

## YOLOv8m Trainieren

YOLO ist die praktischste Video-Basis: schnelle One-Stage-Detection, gute Ultralytics-Artefakte und direkte Weiterverwendung in `step08_blur_video.py`.

In [ ]:
yolo_smoke_cmd = [
    PYTHON, LAB / 'step02_train_ultralytics_detector.py',
    '--family', 'yolo', '--base', 'yolov8m.pt',
    '--epochs', SMOKE_EPOCHS, '--batch', BATCH_SIZE, '--reduction', SMOKE_REDUCTION,
    '--imgsz', IMGSZ, '--train-limit', 0, '--val-limit', 0, '--workers', WORKERS,
]
yolo_deep_cmd = [
    PYTHON, LAB / 'step02_train_ultralytics_detector.py',
    '--family', 'yolo', '--base', 'yolov8m.pt',
    '--epochs', DEEP_EPOCHS, '--batch', BATCH_SIZE, '--reduction', DEEP_REDUCTION,
    '--imgsz', IMGSZ, '--train-limit', 0, '--val-limit', 0, '--workers', WORKERS,
]
print('# YOLO Smoke')
show_command(yolo_smoke_cmd)
print('\n# YOLO Kandidatenlauf')
show_command(yolo_deep_cmd)

## Faster R-CNN Trainieren

Faster R-CNN ist die klassische Two-Stage-Baseline. Es ist qualitativ interessant, aber in der Video-Pipeline schwerfaelliger als YOLO und im Smoke-Vergleich langsamer als FCOS/RetinaNet.

In [ ]:
faster_cmd = [
    PYTHON, LAB / 'step03_train_torchvision_detector.py',
    '--kind', 'fasterrcnn',
    '--epochs', SMOKE_EPOCHS, '--batch', BATCH_SIZE, '--reduction', SMOKE_REDUCTION,
    '--workers', WORKERS, '--save-every', SMOKE_EPOCHS,
]
show_command(faster_cmd)

## RetinaNet Trainieren

RetinaNet ist ein One-Stage-Detector mit Focal Loss. Er ist als Vergleich wichtig, fiel aber im Smoke-Vergleich wegen niedrigerem Recall hinter FCOS und YOLO zurueck.

In [ ]:
retina_cmd = [
    PYTHON, LAB / 'step03_train_torchvision_detector.py',
    '--kind', 'retinanet',
    '--epochs', SMOKE_EPOCHS, '--batch', BATCH_SIZE, '--reduction', SMOKE_REDUCTION,
    '--workers', WORKERS, '--save-every', SMOKE_EPOCHS,
]
show_command(retina_cmd)

## FCOS Trainieren

FCOS ist ein anchor-free One-Stage-Detector. Im breiten Smoke-Vergleich hatte FCOS den besten Recall und wurde deshalb zusammen mit YOLO fuer den laengeren Kandidatenvergleich gewaehlt.

In [ ]:
fcos_smoke_cmd = [
    PYTHON, LAB / 'step03_train_torchvision_detector.py',
    '--kind', 'fcos',
    '--epochs', SMOKE_EPOCHS, '--batch', BATCH_SIZE, '--reduction', SMOKE_REDUCTION,
    '--workers', WORKERS, '--save-every', SMOKE_EPOCHS,
]
fcos_deep_cmd = [
    PYTHON, LAB / 'step03_train_torchvision_detector.py',
    '--kind', 'fcos',
    '--epochs', DEEP_EPOCHS, '--batch', BATCH_SIZE, '--reduction', DEEP_REDUCTION,
    '--workers', WORKERS, '--save-every', DEEP_EPOCHS,
]
print('# FCOS Smoke')
show_command(fcos_smoke_cmd)
print('\n# FCOS Kandidatenlauf')
show_command(fcos_deep_cmd)

## Evaluation Nicht Hier

Alle Modellvergleiche, COCO-Baselines, Metriken, Ressourcen und Grafiken liegen in `step06_evaluation_lab.ipynb`. Dadurch werden Trainingserklaerung und Bewertung nicht doppelt gepflegt.